# Case Study: Bayesian logistic regression with a Laplace prior via SOUL with PAIES step and the stretch move.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"


In [ ]:
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))  
sys.path.append(project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
import pickle
from algorithms import sig, log_p_laplace, so_stretch_fast_decay
#pip_ula, prox_pgd, proximal_map_laplace_iteration_total, pipgla, proximal_map_laplace_approx_total
#from testlaura import log_p_laplace, soul_stretch_fast_decay, run_single_soul_stretch_decay
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

os.chdir(project_root)

# Obtain synthetic dataset for Laplace prior case

In [ ]:
from scipy.stats import laplace, bernoulli
# Data and design matrix
np.random.seed(3)
design_matrix = np.random.uniform(low = -1.0, high = 1.0, size = (900, 50))
x_unknown = laplace.rvs(loc = -4, size = (50, 1))
parameter_bernoulli = sig(np.matmul(design_matrix, x_unknown))
data_experiment = bernoulli.rvs(np.array(parameter_bernoulli[:, 0]), size = 900)
labels = np.expand_dims(data_experiment, axis=1)
theta_true = -4

# SOUL PAIES algorithm with stretch move

In [ ]:
# One run test
# --- SOUL Optimization Setup ---
D = 50       # Dimensionality of latent variables (x)
T = 1100      # Outer optimization steps
M = 25       # MH steps per outer loop
B = 5       # Burn-in steps
delta_step = 0.08
#proposal_std = 0.05
b_scale = 1.0 # Scale parameter for the Laplace prior

# Initializations

N = 70 #100 # Number of walkers
x0_N = np.random.normal(0, 0.1, (D, N)) # Initialize x0_N with some slight noise
#x0_M = np.zeros((D, 1))   # Initial latent variables vector
th0 = np.array([[-15.0]]) 


# Run the adapted algorithm
th_list, x_samples = so_stretch_fast_decay(
    log_p=log_p_laplace,
    th0=th0,
    x0_N=x0_N,
    y_l=design_matrix,
    y_f=labels,
    T=T,
    M=M,
    B=B,
    #D=D,
    delta_step=delta_step,
    #proposal_std=proposal_std,
    b=b_scale,
    gamma=0.9999995
)

print("Final estimated theta:", th_list[-1][0, 0])
print("True theta (loc parameter): -4.0")

In [ ]:
#  Without parallelisation

# --- Experiment Parameters ---
T = 1000  # Outer steps 
B = 5               
M = 25 

# Tuning hyper-parameters for SOUL-MH
delta_step = 0.08    # Theta learning rate
b_scale = 1.0        # Laplace prior scale
gamma_val = 0.9999995

fig = plt.figure(figsize=(10, 6))
theta_stretch = []
X_stretch = []

for run_idx in range(7):
    # Randomly initialize theta0 between -15 and 10
    theta0_val = np.random.randint(-15, 10)
    th0 = np.array([[float(theta0_val)]])
    
    # Initialize X0 drawn from Gaussian centered at theta0, shape (D, M)
    # where D is feature dimension (50 from Paula's dataset)
    D = design_matrix.shape[1]
    X0_M = np.random.normal(loc=theta0_val, scale=1.0, size=(D, M))
    
    th_list, x_values = so_stretch_fast_decay(
    log_p=log_p_laplace,
    th0=th0,
    x0_N=x0_N,
    y_l=design_matrix,
    y_f=labels,
    T=T,
    M=M,
    B=B,
    #D=D,
    delta_step=delta_step,
    #proposal_std=proposal_std,
    b=b_scale,
    gamma=gamma_val
)
    
    # Extract flattened array of theta trajectory across iterations
    th_trajectory = np.array([t[0, 0] for t in th_list])
    
    theta_stretch.append(th_trajectory)
    X_stretch.append(x_values)
    
    # Plot trajectory for this run
    plt.plot(th_trajectory, label=f'Run {run_idx + 1} (Init $\\theta_0={theta0_val}$)')

# Reference line for ground truth mean
plt.axhline(y=np.mean(x_unknown), color='black', linestyle='dashed', label='True Mean $\\theta$')

plt.title('SOSTRETCH Convergence Across Multiple Initializations')
plt.xlabel('Iteration (T)')
plt.ylabel('Theta Estimate ($\\theta$)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.show()

The relative error

In [ ]:
# Convert list of trajectories into a 2D numpy array of shape (num_runs, T + 1)
theta_stretch_arr = np.array(theta_stretch)

# Compute relative error trajectory for each run
relative_error_trajectories = np.abs(theta_stretch_arr - theta_true) / np.abs(
    theta_true
)

# Mean relative error across all runs per iteration step
mean_relative_error_per_step = np.mean(relative_error_trajectories, axis=0)


plt.figure(figsize=(10, 5))
for idx, err_traj in enumerate(relative_error_trajectories):
    plt.plot(err_traj, alpha=0.4, label=f"Run {idx + 1}")

plt.plot(
    mean_relative_error_per_step,
    color="black",
    linewidth=2,
    label="Mean Relative Error",
)
plt.yscale("log")  # Log scale highlights convergence speed clearly
plt.title("SOSTRETCH Relative Error Convergence")
plt.xlabel("Iteration (T)")
plt.ylabel("Relative Error (Log Scale)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

# Errors with the same $\theta_0$

In [ ]:
# --- Experiment Parameters ---
T = 1200  # Outer steps 
B = 5               
M = 25 
D=50
N=76

# Tuning hyper-parameters for SOUL-MH
delta_step = 0.08    # Theta learning rate
b_scale = 1.0        # Laplace prior scale
gamma_val = 0.9999995

x0_N = np.random.normal(0, 0.1, (D, N)) # Initialize x0_N with some slight noise


fig = plt.figure(figsize=(10, 6))
theta_stretch_same = []
X_stretch_same = []

for run_idx in range(5):
    theta0_val = 3.0
    th0 = np.array([[float(theta0_val)]])
    
    D = design_matrix.shape[1]
    X0_M = np.random.normal(loc=theta0_val, scale=1.0, size=(D, M))
    
    th_list, x_values = so_stretch_fast_decay(
    log_p=log_p_laplace,
    th0=th0,
    x0_N=x0_N,
    y_l=design_matrix,
    y_f=labels,
    T=T,
    M=M,
    B=B,
    #D=D,
    delta_step=delta_step,
    #proposal_std=proposal_std,
    b=b_scale,
    gamma=gamma_val
    )
    
    # Extract flattened array of theta trajectory across iterations
    th_trajectory_same = np.array([t[0, 0] for t in th_list])
    
    theta_stretch_same.append(th_trajectory_same)
    X_stretch_same.append(x_values)
    

# Convert list of trajectories into a 2D numpy array of shape (num_runs, T + 1)
theta_stretch_arr_same = np.array(theta_stretch_same)

# Compute relative error trajectory for each run
relative_error_trajectories_same = np.abs(theta_stretch_arr_same - theta_true) / np.abs(
    theta_true
)

# Mean relative error across all runs per iteration step
mean_relative_error_per_step_same = np.mean(relative_error_trajectories_same, axis=0)


plt.figure(figsize=(10, 5))
for idx, err_traj in enumerate(relative_error_trajectories_same):
    plt.plot(err_traj, alpha=0.4, label=f"Run {idx + 1}")

plt.plot(
    mean_relative_error_per_step_same,
    color="black",
    linewidth=2,
    label="Mean Relative Error",
)
plt.yscale("log")  # Log scale highlights convergence speed clearly
plt.title("SOSTRETCH Relative Error Convergence (theta0 = 3.0)")
plt.xlabel("Iteration (T)")
plt.ylabel("Relative Error (Log Scale)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

# Save and recover the data 

In [ ]:
# Save trajectories, ground truth, and precalculated errors
np.savez_compressed(
    "so_stretch_results.npz",
    T=T,
    B=B,
    M=M,
    delta_step=delta_step,
    theta_true=theta_true,
    theta_stretch_arr=theta_stretch_arr,
    gamma_val = gamma_val,
    b_scale = b_scale,
    relative_error_trajectories=relative_error_trajectories,
    mean_relative_error_per_step=mean_relative_error_per_step,
    X_stretch_same=X_stretch_same,
    theta_stretch_arr_same= theta_stretch_arr_same,
    mean_relative_error_per_step_same  = mean_relative_error_per_step_same,
    relative_error_trajectories_same=relative_error_trajectories_same,

)

print("Data saved successfully to 'so_stretch_results.npz'")

In [ ]:
# Load compressed data
data = np.load("so_stretch_results.npz")

T = data["T"]
theta_true = data["theta_true"]
theta_stretch_arr = data["theta_stretch_arr"]
relative_error_trajectories = data["relative_error_trajectories"]
mean_relative_error_per_step = data["mean_relative_error_per_step"]

# Plot 1: Theta Convergence Trajectories ---
plt.figure(figsize=(10, 6))

for idx, th_trajectory in enumerate(theta_stretch_arr):
    plt.plot(th_trajectory, alpha=0.7, label=f"Run {idx + 1}")

plt.axhline(
    y=theta_true,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=r"$\theta_{gen}$",
)

plt.title("SOStretch Evolution Across Multiple Initializations", fontsize=14, pad=12)
plt.xlabel("Iteration ($T$)", fontsize=12)
plt.ylabel(r"Theta Estimate ($\theta$)", fontsize=12)
plt.legend(loc="upper right", frameon=True, fontsize=10)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Plot 2: Relative Error Convergence (Log Scale) ---
plt.figure(figsize=(10, 5))

for idx, err_traj in enumerate(relative_error_trajectories):
    plt.plot(err_traj, alpha=0.3, label=f"Run {idx + 1}")

plt.plot(
    mean_relative_error_per_step,
    color="black",
    linewidth=2,
    label="Mean Relative Distance",
)

plt.yscale("log")
plt.title("SOStretch Relative Error Evolution", fontsize=14, pad=12)
plt.xlabel("Iteration ($T$)", fontsize=12)
plt.ylabel("Relative Distance (Log Scale)", fontsize=12)
plt.legend(loc="upper right", frameon=True, fontsize=10)
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()